In [31]:
import os
import re
import sys
import pandas as pd


META_SPLIT_CSV = "/home/sagemaker-user/files/workspace_files/Foundation_exploration/ecg-fm/splits/meta_split_mimic_iv_ecg.csv"
MACHINE_MEASUREMENTS_CSV = "/home/sagemaker-user/files/workspace_files/MIMIC/MIMIC-ECG/physionet.org/files/mimic-iv-ecg/1.0/machine_measurements.csv"

FAIRSEQ_SIGNALS_ROOT = "/home/sagemaker-user/files/workspace_files/Foundation_exploration/fairseq-signals"
sys.path.insert(0, os.path.join(FAIRSEQ_SIGNALS_ROOT, "labeler"))

LABELER_DIR = "/home/sagemaker-user/files/workspace_files/Foundation_exploration/ecg-fm/data/mimic_iv_ecg/labeler"

OUT_CSV = "./mimic_iv_ecg_test10k_afib.csv"

# Sampling
N_TARGET = 10_000
SEED = 0

# Label name for AFIB in ECG-FM labeler
AF_LABEL_NAME = "Atrial fibrillation"

# Optional cache (can be any writable path)
LABELER_CACHE_PKL = "/tmp/labeler_res.pkl"

In [32]:
meta = pd.read_csv(META_SPLIT_CSV)
meta.columns = meta.columns.str.strip()

# Clean split values to avoid hidden spaces
meta["split"] = meta["split"].astype(str).str.strip()

print("meta shape:", meta.shape)
print("meta columns:", list(meta.columns))
print(meta["split"].value_counts())
meta.head(1)

meta shape: (787677, 3)
meta columns: ['source_path', 'save_file', 'split']
split
train    630178
valid     79129
test      78370
Name: count, dtype: int64


,source_path,save_file,split
0,files/p1000/p10000032/s40689238/40689238,mimic_iv_ecg_p1000_p10000032_s40689238_4068923...,train


In [33]:
# Select test pool
test_pool = meta[meta["split"] == "test"].copy()
print("Total test rows:", len(test_pool))

# Extract study_id from save_file:
# Example save_file:
# mimic_iv_ecg_p1000_p10000032_s40689238_40689238.mat -> 40689238
def extract_study_id_from_save_file(s):
    m = re.search(r"_([0-9]+)\.mat$", str(s))
    return m.group(1) if m else None

test_pool["study_id"] = test_pool["save_file"].apply(extract_study_id_from_save_file)
test_pool = test_pool.dropna(subset=["study_id"]).copy()
test_pool["study_id"] = test_pool["study_id"].astype(str)

# Remove duplicates on study_id just in case
test_pool = test_pool.drop_duplicates(subset=["study_id"]).copy()

print("Test pool after parsing study_id:", len(test_pool))
test_pool[["source_path", "save_file", "split", "study_id"]].head(1)

Total test rows: 78370
Test pool after parsing study_id: 78370


,source_path,save_file,split,study_id
8,files/p1000/p10000635/s40067704/40067704,mimic_iv_ecg_p1000_p10000635_s40067704_4006770...,test,40067704


In [34]:
mm = pd.read_csv(MACHINE_MEASUREMENTS_CSV)
mm.columns = mm.columns.str.strip()

if "study_id" not in mm.columns:
    raise ValueError("machine_measurements.csv does not contain a 'study_id' column. Please check the file.")

mm["study_id"] = mm["study_id"].astype(str)

# Find all report_* columns
report_cols = [c for c in mm.columns if c.startswith("report_")]
if len(report_cols) == 0:
    raise ValueError("No columns starting with 'report_' were found in machine_measurements.csv.")

print("Found report columns:", report_cols)

# Concatenate report columns into one string
mm["report_text"] = (
    mm[report_cols]
      .fillna("")
      .agg(" ".join, axis=1)
      .str.replace(r"\s+", " ", regex=True)
      .str.strip()
)

mm_reports = mm[["study_id", "report_text"]].copy()

print("machine_measurements rows:", len(mm_reports))
mm_reports.head(1)

/tmp/ipykernel_27492/2493979991.py:1: DtypeWarning: Columns (16,17,18,19,20,21) have mixed types. Specify dtype option on import or set low_memory=False.
  mm = pd.read_csv(MACHINE_MEASUREMENTS_CSV)


Found report columns: ['report_0', 'report_1', 'report_2', 'report_3', 'report_4', 'report_5', 'report_6', 'report_7', 'report_8', 'report_9', 'report_10', 'report_11', 'report_12', 'report_13', 'report_14', 'report_15', 'report_16', 'report_17']
machine_measurements rows: 800035


,study_id,report_text
0,40689238,Sinus rhythm Possible right atrial abnormality...


In [35]:
# Merge report_text into test pool
merged = test_pool.merge(mm_reports, on="study_id", how="left")

# Keep rows with non-empty report_text
merged["report_text"] = merged["report_text"].fillna("").astype(str)
merged_valid = merged[merged["report_text"].str.len() > 0].copy()

print("Merged rows:", len(merged))
print("Rows with valid report_text:", len(merged_valid))

if len(merged_valid) < N_TARGET:
    raise ValueError(
        f"Not enough samples with valid reports in test split. "
        f"Need {N_TARGET}, but only found {len(merged_valid)}."
    )

# Sample exactly N_TARGET from those with valid reports
test_df = merged_valid.sample(n=N_TARGET, random_state=SEED).copy()
print("Final test_df rows (with report_text):", len(test_df))

test_df[["study_id", "source_path", "save_file", "split", "report_text"]].head(1)

Merged rows: 78370
Rows with valid report_text: 78369
Final test_df rows (with report_text): 10000


,study_id,source_path,save_file,split,report_text
3787,42341564,files/p1051/p10512468/s42341564/42341564,mimic_iv_ecg_p1051_p10512468_s42341564_4234156...,test,Sinus rhythm. Possible inferior infarct - age ...


In [36]:
# Make labeler importable
sys.path.append("/home/sagemaker-user/files/workspace_files/Foundation_exploration/ecg-fm/labeler")
from pattern_labeler import PatternLabelerConfig, PatternLabeler
from preprocess import preprocess_texts

cfg = PatternLabelerConfig.from_json(LABELER_DIR)
labeler = PatternLabeler(cfg)

print("Labeler loaded.")

Labeler loaded.


In [42]:
import numpy as np
import pandas as pd

# -----------------------------
# Preprocess texts (robust: remove NaN/float)
# -----------------------------

# 1) Start from report_text and force it to pure python strings
report_series = test_df["report_text"].values  # numpy array for fastest type checks

clean_reports = []
for x in report_series:
    if x is None:
        clean_reports.append("")
    elif isinstance(x, float) and np.isnan(x):
        clean_reports.append("")
    else:
        s = str(x)
        s = " ".join(s.split())  # normalize whitespace
        clean_reports.append(s)

report_series = pd.Series(clean_reports)

# Drop empty reports (safety)
mask = report_series.str.len() > 0
test_df = test_df.loc[mask].copy().reset_index(drop=True)
report_series = report_series.loc[mask].reset_index(drop=True)

# 2) Run preprocess_texts (may output NA), then force back to pure python strings
texts = preprocess_texts(report_series)

clean_texts = []
for x in texts.values:
    if x is None:
        clean_texts.append("")
    elif isinstance(x, float) and np.isnan(x):
        clean_texts.append("")
    else:
        clean_texts.append(str(x))

# IMPORTANT: idx becomes study_id
texts = pd.Series(clean_texts, index=test_df["study_id"].astype(str).values)
texts.name = "text"

# Final sanity check: no non-str allowed
assert texts.apply(lambda z: isinstance(z, str)).all(), "texts contains non-str values (unexpected)"

# Remove old cache if exists
if os.path.exists(LABELER_CACHE_PKL):
    os.remove(LABELER_CACHE_PKL)

# Run labeler (start with no cache to avoid stale pickle issues)
labeler_res = labeler(texts=texts.copy(), restore_path=None)

labels_flat = labeler_res.labels_flat.copy()
print("labels_flat rows:", len(labels_flat))
labels_flat.head(10)

Pre replacements...
Replacements...
Replacements - Unbordered
Replacements - Bordered
Replacements - Regex
Post replacements...
Replacing 'pacing' conjunct patterns...


100%|██████████| 1316/1316 [00:35<00:00, 36.69it/s]


Performed replacements in 127 entries.
Replacing 'pacer entities' conjunct patterns...


100%|██████████| 40/40 [00:01<00:00, 33.82it/s]


Performed replacements in 5 entries.
Replacing 'wave abnormality' conjunct patterns...


...

100%|██████████| 14/14 [00:00<00:00, 73.04it/s]


Creating entities and attributing descriptors...
Parsing patterns into entities...


100%|██████████| 10000/10000 [00:18<00:00, 550.93it/s]


Applying uncertainty...
Dropped 0 entities with absence descriptors.
Recursively extract super entities...
Parsing traveling descriptors...
Parsing attached descriptors...
Recursively extract super entities...
Creating compound entities...
Created 1038 compound entities.
Recursively extract super entities...
labels_flat rows: 122483


,name,prob
sample,,
40000538,Abnormal electrocardiogram,1.0
40000538,Hypertrophy,1.0
40000538,Hypertrophy - Possible,1.0
40000538,Hypertrophy - Probable,1.0
40000538,Ischemia,0.7
40000538,Ischemia - Probable,1.0
40000538,Left axis deviation,1.0
40000538,Left ventricular hypertrophy,1.0
40000538,ST wave abnormality,1.0


In [43]:
# -----------------------------
# Sanity check: AFIB label exists
# -----------------------------

AF_LABEL_NAME = "Atrial fibrillation"

top_labels = labels_flat["name"].value_counts().head(30)
print("Top parsed labels:")
print(top_labels)

if AF_LABEL_NAME not in labels_flat["name"].unique():
    raise ValueError(
        f"'{AF_LABEL_NAME}' not found in labeler output. "
        f"Check LABELER_DIR config and report_text construction."
    )

af_count = (labels_flat["name"] == AF_LABEL_NAME).sum()
print(f"AFIB mentions in labels_flat rows: {af_count}")

Top parsed labels:
name
Sinus rhythm                       8068
Abnormal electrocardiogram         4679
Arrhythmia                         3729
T wave abnormality                 3681
Conduction defect                  3301
Ischemia                           3085
ST wave abnormality                2391
Injury                             2370
Infarction                         2290
Borderline electrocardiogram       2204
High heart rate                    2148
Tachycardia                        2148
Conduction block                   2123
ST-T wave abnormality              1867
T wave abnormality - Multiple      1692
Supraventricular rhythm            1572
Intraventricular block             1537
Left axis deviation                1484
Normal electrocardiogram           1433
Atrial rhythm                      1372
Aberrant ventricular conduction    1307
Bradycardia                        1273
Low heart rate                     1273
Ectopy                             1245
Ectopic contract

In [46]:
# -----------------------------
# Build AFIB-positive study_id set from labels_flat
# (handles cases where 'idx' is not a column)
# -----------------------------

AF_LABEL_NAME = "Atrial fibrillation"

print("labels_flat columns:", list(labels_flat.columns))
print(labels_flat.head(3))

# Decide where the sample id lives
candidate_cols = ["idx", "study_id", "text_id", "text_idx", "sample_id", "id"]
id_col = next((c for c in candidate_cols if c in labels_flat.columns), None)

if id_col is not None:
    # ID stored as a column
    ids_for_af = labels_flat.loc[labels_flat["name"] == AF_LABEL_NAME, id_col].astype(str)
    id_source = f"column '{id_col}'"
else:
    # ID stored in the index
    ids_for_af = labels_flat.loc[labels_flat["name"] == AF_LABEL_NAME].index.astype(str)
    id_source = "labels_flat.index"

af_ids = set(ids_for_af.unique())
print("Using id source:", id_source)
print("AFIB positive id count:", len(af_ids))

labels_flat columns: ['name', 'prob']
                                name  prob
sample                                    
40000538  Abnormal electrocardiogram   1.0
40000538                 Hypertrophy   1.0
40000538      Hypertrophy - Possible   1.0
Using id source: labels_flat.index
AFIB positive id count: 1021


In [47]:
# -----------------------------
# Create binary AFIB label for each study_id in test_df
# -----------------------------

test_df["study_id"] = test_df["study_id"].astype(str)
test_df["afib"] = test_df["study_id"].isin(af_ids).astype(int)

print("AFIB label distribution:")
print(test_df["afib"].value_counts(dropna=False))

test_df[["study_id", "afib", "source_path", "save_file"]].head(10)

AFIB label distribution:
afib
0    8979
1    1021
Name: count, dtype: int64


,study_id,afib,source_path,save_file
0,42341564,0,files/p1051/p10512468/s42341564/42341564,mimic_iv_ecg_p1051_p10512468_s42341564_4234156...
1,40965333,1,files/p1721/p17211916/s40965333/40965333,mimic_iv_ecg_p1721_p17211916_s40965333_4096533...
2,44766763,1,files/p1832/p18323186/s44766763/44766763,mimic_iv_ecg_p1832_p18323186_s44766763_4476676...
3,40441658,0,files/p1002/p10022863/s40441658/40441658,mimic_iv_ecg_p1002_p10022863_s40441658_4044165...
4,41908718,0,files/p1720/p17203862/s41908718/41908718,mimic_iv_ecg_p1720_p17203862_s41908718_4190871...
5,49892505,0,files/p1837/p18377213/s49892505/49892505,mimic_iv_ecg_p1837_p18377213_s49892505_4989250...
6,40121531,0,files/p1517/p15170481/s40121531/40121531,mimic_iv_ecg_p1517_p15170481_s40121531_4012153...
7,44429082,0,files/p1324/p13242444/s44429082/44429082,mimic_iv_ecg_p1324_p13242444_s44429082_4442908...
8,47504035,0,files/p1008/p10082560/s47504035/47504035,mimic_iv_ecg_p1008_p10082560_s47504035_4750403...
9,41048673,0,files/p1739/p17394638/s41048673/41048673,mimic_iv_ecg_p1739_p17394638_s41048673_4104867...


In [48]:
# -----------------------------
# Save final 10k dataset (or slightly less if empty reports were dropped)
# -----------------------------

OUT_CSV = "./mimic_iv_ecg_test10k_afib.csv"

final_df = test_df[["study_id", "source_path", "save_file", "split", "afib"]].copy()
final_df.to_csv(OUT_CSV, index=False)

print("Saved:", OUT_CSV)
print("Final shape:", final_df.shape)
final_df.head(5)

Saved: ./mimic_iv_ecg_test10k_afib.csv
Final shape: (10000, 5)


,study_id,source_path,save_file,split,afib
0,42341564,files/p1051/p10512468/s42341564/42341564,mimic_iv_ecg_p1051_p10512468_s42341564_4234156...,test,0
1,40965333,files/p1721/p17211916/s40965333/40965333,mimic_iv_ecg_p1721_p17211916_s40965333_4096533...,test,1
2,44766763,files/p1832/p18323186/s44766763/44766763,mimic_iv_ecg_p1832_p18323186_s44766763_4476676...,test,1
3,40441658,files/p1002/p10022863/s40441658/40441658,mimic_iv_ecg_p1002_p10022863_s40441658_4044165...,test,0
4,41908718,files/p1720/p17203862/s41908718/41908718,mimic_iv_ecg_p1720_p17203862_s41908718_4190871...,test,0
